# Crash!💥 Boom!💥 Bang! 💥🚗🚙💥🚗
**A SQL based exploratory data analysis of traffic accidents in Okinawa, Japan**
  
---   
# <span style="color:#0279f0;">Part 1: Data Quality & Integrity Check</span>
---


### **1.1 Getting to know the Main Dataset**
**Choice of Tool:**
- Pandas; Since the csv file is around 50MB, loading the data into a dataframe for the quality and integrity check will be more efficient than using SQL queries.  

**Tasks:**
- Load main data set
- Retrieve column names and data types
- Count total number of records
- Check if there are any null values

##### 👩🏻‍💻  *<span style="color:#0279f0">Loading Main Dataset, Null Check & Record Count...</span>*

In [23]:
import pandas as pd

from config import DATA_SRC

# Read the CSV directly into pandas data frame
df = pd.read_csv(DATA_SRC)

# Count total no. of records in dataset
print("--------------------")
print(f"✅ Data loaded successfully.")
print(f"✅ Total records: {len(df)}")
print("--------------------")

# Column information
col_info = pd.DataFrame({
    'Column Name': df.columns,
    'Data Type': df.dtypes.values,
    'Non-Null Count': df.count().values
})
print("Column Information:")
print(col_info.to_string(index=False))
print("--------------------")

#Quick validation: Show first few rows
print("First 5 rows:")
print(df.head())

--------------------
✅ Data loaded successfully.
✅ Total records: 290895
--------------------
Column Information:
                         Column Name Data Type  Non-Null Count
                       data_category     int64          290895
                     prefecture_code     int64          290895
                 police_station_code     int64          290895
                           report_id     int64          290895
               accident_details_code     int64          290895
                          fatalities     int64          290895
                     injured_persons     int64          290895
                          route_code     int64          290895
                       location_code     int64          290895
                   municipality_code     int64          290895
                      occurence_year     int64          290895
                     occurence_month     int64          290895
                      occurence_date     int64          290895
    

Total no. of records = 290895.  
Total no. of non-null records for each field = 290895.  
 **Great! No Null Records Found!**  
#### <span style="color:green">Null Check Results: Passed ✅</span>
..........



### **1.2 Validate Dates and Prefecture (City) Codes**   

**Tasks:**
- Ensure the date range of the records are valid for the EDA (year 2024 only)
- Ensure the data is Okinawa-centric and does not contain data for other prefectures
- Okinawa's prefecture_code is 97 (refer to prefecture code documentation)

##### 👩🏻‍💻 *<span style="color:#0279f0">Validating date range and prefecture code...</span>*

In [24]:
# ========================================
#Check date ranges found in the data set
# ========================================

#Count no, of unique dates in entire data set
unique_days = df[['occurence_month','occurence_day']].drop_duplicates().shape[0]

#Ger date ranges
results_date_validation = pd.DataFrame({
    'earliest_year': [df['occurence_year'].min()],
    'latest_year': [df['occurence_year'].max()],
    'earliest_month': [df['occurence_month'].min()],
    'latest_month': [df['occurence_month'].max()],
    'earliest_date': [df['occurence_date'].min()],
    'latest_date': [df['occurence_date'].max()],
    'unique_days': [unique_days]
})

#print results
print("\n\n----------------------------------------")
print("📆 DATE VALIDATION RESULTS")
print("----------------------------------------")
display(results_date_validation)


KeyError: "['occurence_day'] not in index"

⚠️ Earliest year: 2019  
⚠️ Total unique days: 709  
⚠️ The dataset contains records prior to 2024.

As this EDA focuses on accidents that occurred in 2024, records prior to 2024 will be excluded from subsequent analyses. Before doing so, however, we will briefly examine these older records to verify that they are valid observations rather than data-quality issues.

#### <span style="color:red">Results: Invalid Dates Detected ⚠️</span>

##### 👩🏻‍💻 *<span style="color:#0279f0">Examining Records with Invalid Dates...</span>*

In [ ]:
# ========================================
#Check accident records distribution by year
# ========================================


#Group by occurence_year and count no. of records
yearly_count = df.groupby('occurence_year').size().reset_index(name='accident_count')

#Count total no. of records
total_count = len(df) 

#Compute percentage distribution of records by year
yearly_count['percentage'] = (yearly_count['accident_count'] / total_count * 100).round(2)

#Sort results by year
yearly_count = yearly_count.sort_values('occurence_year').reset_index(drop=True)

#print results
print("\n\n-------------------------------------------------")
print("📆 Percentage Distribution Of Records By Year")
print("-------------------------------------------------")
display(yearly_count)

# ========================================
# Review records prior to 2024
# ========================================

# Columns to inspect
cols_to_review = [
    'occurence_year',
    'occurence_month',
    'occurence_date',
    'occurence_hour',
    'fatalities',
    'injured_persons',
    'weather_code',
    'accident_details_code',
    'location_code'
]

# Define sample sizes per year
sample_sizes = {
    2019: 1,
    2020: 1,
    2021: 3,
    2022: 10,
    2023: 10
}

# Collect samples
samples_by_year = {}

for year, size in sample_sizes.items():
    df_year = df[df['occurence_year'] == year]
    if len(df_year) >= size:
        sample = df_year[cols_to_review].sample(n=size, random_state=42)  # fixed seed for reproducibility
    else:
        sample = df_year[cols_to_review]  # take all if fewer exist
    samples_by_year[year] = sample
    print(f"\n🔍 Year {year} - {len(sample)} random records:")
    display(sample)

# ========================================
# Summary of records prior to 2024
# ========================================
summary_pre_2024 = (

    df[df['occurence_year'] < 2024]
    .groupby('occurence_year')

    .agg(
        total_records=('occurence_year', 'size'),
        unique_months=('occurence_month', 'nunique'),
        unique_dates=('occurence_date', 'nunique'),
        unique_hours=('occurence_hour', 'nunique')
    )
    .reset_index()
)

# ========================================
# Summary of 2024 records
# ========================================
summary_2024 = (

    df[df['occurence_year'] == 2024]
    .groupby('prefecture_code')

    .agg(
        total_records=('occurence_year', 'size'),
        unique_months = ('occurence_month', 'nunique'),
        unique_dates = ('occurence_date', 'nunique'),
        unique_hours = ('occurence_hour', 'nunique'),
        unique_pref_code = ('prefecture_code', 'nunique'),
        unique_municipality_code = ('municipality_code', 'nunique')

    )

    .reset_index()
)


#print results 
print("\n\n-------------------------------------------------")
print("📆  Summary of Pre-2024 Records")
print("-------------------------------------------------")


display(summary_pre_2024)


print("\n\n-------------------------------------------------")
print("📆 Summary of 2024 Records by Prefecture Code")
print("-------------------------------------------------")

display(summary_2024)



-------------------------------------------------
📆 Percentage Distribution Of Records By Year
-------------------------------------------------


,occurence_year,accident_count,percentage
0,2019,1,0.00
1,2020,1,0.00
2,2021,3,0.00
3,2022,37,0.01
4,2023,8477,2.91
5,2024,282376,97.07



🔍 Year 2019 - 1 random records:


,occurence_year,occurence_month,occurence_date,occurence_hour,fatalities,injured_persons,weather_code,accident_details_code,location_code
27739,2019,7,2,7,0,1,1,2,0



🔍 Year 2020 - 1 random records:


...

,occurence_year,occurence_month,occurence_date,occurence_hour,fatalities,injured_persons,weather_code,accident_details_code,location_code
2338,2023,9,27,18,0,1,1,2,0
48748,2023,12,27,3,0,1,1,2,0
15855,2023,12,19,8,0,1,1,2,0
526,2023,11,22,18,0,1,2,2,0
18912,2023,10,15,1,0,1,1,2,170
37890,2023,12,27,18,0,1,1,2,0
37665,2023,11,22,20,0,1,1,2,0
975,2023,1,2,11,0,1,2,2,150
22374,2023,12,13,17,0,1,1,2,0
5243,2023,12,4,13,0,1,1,2,0




-------------------------------------------------
📆  Summary of Pre-2024 Records
-------------------------------------------------


,occurence_year,total_records,unique_months,unique_dates,unique_hours
0,2019,1,1,1,1
1,2020,1,1,1,1
2,2021,3,3,3,3
3,2022,37,11,16,17
4,2023,8477,12,31,24




-------------------------------------------------
📆 Summary of 2024 Records by Prefecture Code
-------------------------------------------------


,prefecture_code,total_records,unique_months,unique_dates,unique_hours,unique_pref_code,unique_municipality_code
0,10,6126,12,31,24,1,65
1,11,528,12,31,23,1,18
2,12,719,12,31,24,1,41
3,13,733,12,31,24,1,29
4,14,211,12,31,21,1,15
5,20,2228,12,31,24,1,38
6,21,1362,12,31,24,1,33
7,22,3655,12,31,24,1,38
8,23,959,12,31,24,1,24
9,24,2401,12,31,24,1,35


### 📊 <span style="color:green">Summary of Pre-2024 Records</span>  

- Records exist from 2019 to 2023.
- 2023 data covers all months and hours, suggesting a full year of data.
- 2019–2022 have very few records and look like isolated entries.
- Samples from each year show normal accident data and valid dates.  
- ✅  These records look valid and consistent and appear to be real historical data, not data entry errors.


### 📊 <span style="color:green">Summary of 2024 Records</span>

- Dataset contains nationwide accident data with 51 prefecture codes, not just Okinawa's.
- Okinawa (prefecture_code = 97) accounts for only 2,749 accidents.
- ✅  2024 records look valid and complete.

*Japan has only 47 prefectures, which doesn't match the count of 51 prefecture codes found. But a quick check with the official prefecture_code documentation confirmed that Hokkaido was assigned 5 prefecture codes, each for a different part of Hokkaido. This brings the total number of unique prefecture codes to 51, matching the dataset.*  

### ✅ <span style="color:green;">Next Course of Action:</span>
All data looks valid and complete. However, since this EDA focuses on 2024 accidents in Okinawa, we will need to 
- Filter out only records for Okinawa from the year 2024. 
- Then, identify the primary key.

### **1.3 Final Data Preparation Step**   

**Tasks:**
- Filter out 2024 Okinawa specific data
- Identify the primary key
- Export subset data as csv for analysis

##### 👩🏻‍💻 *<span style="color:#0279f0">Filter, Identify, Export...</span>*

In [ ]:
# ========================================
# Filter for Okinawa (prefecture_code == 97) and year 2024
# ========================================

#create 2024 Okinawa subset
okinawa_2024 = df[(df['prefecture_code'] == 97) & (df['occurence_year'] == 2024)].copy()

#Validate no, of unique days in 2024 subset
count_days = okinawa_2024[['occurence_month', 'occurence_date']].drop_duplicates().shape[0]

print("\n\n-------------------------------------------------")
print(f"🗄️ Filtered {len(okinawa_2024)} records for Okinawa 2024")
print(f"⚠️ Total no. of unique days: {count_days}.")


# Unique date counts in the subset did not match number of calendar days for 2024
print("⚠️ 1 calendar day missing in 2024 Okinawa subset")
# Generate all possible (month, day) for 2024 (leap year)
from datetime import date, timedelta

start_date = date(2024, 1, 1)
end_date = date(2024, 12, 31)
all_dates = []
current = start_date
while current <= end_date:
    all_dates.append((current.month, current.day))
    current += timedelta(days=1)

# Get distinct (month + date) pairs from 2024 Okinawa subset
observed = set(zip(okinawa_2024['occurence_month'], okinawa_2024['occurence_date']))

# Find missing dates
missing = [f"{m:02d}-{d:02d}" for m, d in all_dates if (m, d) not in observed]

print(f"Total calendar days in 2024: {len(all_dates)}")
print(f"Total observed days in 2024 data: {len(observed)}")
print(f"Missing days: {len(missing)}")
print(f"Missing: {missing[:10]}", "..." if len(missing) > 10 else "")

# ========================================
# Identify Primary Key
# ========================================

# Quick check if report_id is unique
is_unique = okinawa_2024['report_id'].is_unique

print("\n-------------------------------------------------")

if is_unique:
    print("✅ Report_id is unique → primary key confirmed")
else:
    print("⚠️ Duplicate report_id found")
    # Show duplicate rows
    dupes = okinawa_2024[okinawa_2024.duplicated('report_id', keep=False)]


    # Sort and display to compare same report_id records side-by-side
    dupes_sorted = dupes.sort_values('report_id')
    print(f"\n🔍 Found {len(dupes)} rows with duplicate report_id values")

    # Display only the specified columns
    cols_to_show = ['report_id', 'police_station_code', 'prefecture_code', 'municipality_code']
    display(dupes_sorted[cols_to_show].head(20))

# ========================================
# Report_id was not the primary key. 
# It appears that the data identifies records
# by using a compound key: report_id + police_station_code
# ========================================

# Check if compound key (report_id + police_station_code) is unique
compound_key = okinawa_2024[['report_id', 'police_station_code']]

if compound_key.duplicated().sum() == 0:
    print("✅ Compound key (report_id, police_station_code) is unique → can serve as primary key")
else:
    dup_count = compound_key.duplicated().sum()
    print(f"⚠️ Found {dup_count} duplicate rows for compound key")



-------------------------------------------------
🗄️ Filtered 2749 records for Okinawa 2024
⚠️ Total no. of unique days: 365.
⚠️ 1 calendar day missing in 2024 Okinawa subset
Total calendar days in 2024: 366
Total observed days in 2024 data: 365
Missing days: 1
Missing: ['06-13'] 

-------------------------------------------------
⚠️ Duplicate report_id found

🔍 Found 2731 rows with duplicate report_id values


,report_id,police_station_code,prefecture_code,municipality_code
22950,1,101,97,201
23055,1,108,97,313
23057,1,109,97,209
23073,1,113,97,208
23097,1,114,97,201
23022,1,105,97,211
23098,2,114,97,212
23023,2,105,97,326
22981,2,102,97,210
23074,2,113,97,208


✅ Compound key (report_id, police_station_code) is unique → can serve as primary key


### ✅ <span style="color:green;">Summary of Characteristics of 2024 Okinawa </span> 
- The 2024 Okinawa Data consists of 2749 record 
- One calendar day (June 13, 2024) has no records. 
- No corrective action is needed for typical aggregation or trend analysis.
- Records are identified using a compound key: report_id and police_station_code

##### 👩🏻‍💻 *<span style="color:#0279f0">Ready To Export 2024 Okinawa Accidents Dataset...</span>*

👉🏼 Note: *The export cell below will only run if EXPORT_COMPLETE = False. This prevents accidental overwrites. To re-export the CSV (e.g., after filtering changes), change the flag to False and run the cell.*  

In [ ]:
# ========================================
# Note: The export cell below will only run if EXPORT_COMPLETE = False. 
# This prevents accidental overwrites. 
# To re-export the CSV (e.g., after filtering changes), change the flag to False and run the cell.
# ========================================

# Set this flag to False if you need to re-export the CSV
EXPORT_COMPLETE = True


if not EXPORT_COMPLETE:
    # Define output path (adjust relative path as needed)
    output_path = '../data/okinawa_2024_accidents.csv'
    
    # Export the csv
    okinawa_2024.to_csv(output_path, index=False, encoding='utf-8')
    print(f"✅ Exported {len(okinawa_2024)} records to {output_path}")
else:
    print("⏭️ Export skipped (EXPORT_COMPLETE flag is True). Set to False to re-export.")

⏭️ Export skipped (EXPORT_COMPLETE flag is True). Set to False to re-export.


---
## 📊 The 2024 Okinawa Traffic Accidents Dataset is now ready for Exploratory Data Analysis.
This concludes the data preparation stage. I'll see you in the next chapter!